# Task 2 — Supply-chain anomaly explainer (QLoRA fine-tune)

<a href="https://colab.research.google.com/github/YOUR_GITHUB/CDAZZDEV-MLE-YourName/blob/main/task2_genai/notebooks/supply_chain_finetune.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem statement
**Input:** one incident record from a supply-chain control tower: event type, industry, severity, SKU, lane, dock, shift, expected vs observed quantity, dwell hours, optional temperature, optional 30-day supplier on-time rate, and an operator note. Records come in three formats (prose, JSON extract, shift log).

**Output:** a JSON object with `likely_cause`, `supporting_signals` (2–4), `immediate_action`, `escalation` (bool), `residual_risk`, `confidence` (0–1).

**Correct:** every signal is present in the record, the cause is consistent with the numbers, and the action is a concrete operational step (for example, quarantine the lot, or tender to the backup carrier).
**Incorrect:** contradicts the numbers (for example, calls a short delivery complete), invents metrics, parties, or costs, gives a generic action ("investigate further"), or is not valid JSON.

**Models:** teacher = Groq `openai/gpt-oss-120b` (data generation only). Student = `Qwen/Qwen2.5-3B-Instruct`, a different model family. Llama 3.3 70B was not available on this Groq account.

Runtime: Colab, T4 or L4 GPU. Secrets needed: `GROQ_API_KEY`, `HF_TOKEN` (Colab → Secrets).

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/YOUR_GITHUB/CDAZZDEV-MLE-YourName.git"  # public repo with task2_genai/data
if not os.path.exists("task2_genai"):
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    os.chdir("repo")
%pip install -q -r requirements-colab.txt
try:
    from google.colab import userdata
    for key in ("GROQ_API_KEY", "HF_TOKEN"):
        os.environ[key] = userdata.get(key)
except ImportError:
    from dotenv import load_dotenv; load_dotenv()
print("cwd:", os.getcwd(), "| keys set:", [k for k in ("GROQ_API_KEY", "HF_TOKEN") if os.environ.get(k)])

## 2A — Dataset: teacher prompt, splits, and diversity

In [ ]:
import json, re, collections
import pandas as pd, matplotlib.pyplot as plt
from task2_genai.src.diversity import diversity_report

DATA = "task2_genai/data"
read = lambda name: [json.loads(line) for line in open(f"{DATA}/{name}.jsonl")]
splits = {s: read(s) for s in ("train", "val", "test")}
meta = {s: read(f"{s}_meta") for s in splits}
total = sum(map(len, splits.values()))
print({s: f"{len(v)} ({len(v)/total:.0%})" for s, v in splits.items()}, "total", total)
print("\n--- Teacher system prompt (used only for data generation) ---\n")
print(open("task2_genai/prompts/teacher_system.txt").read())
print("--- Example chat record (Qwen chat format) ---")
print(json.dumps(splits["train"][0], indent=1)[:1500])

In [ ]:
records = [{"user_text": r["messages"][1]["content"], "event_type": m["event_type"], "industry": m["industry"]}
           for s in splits for r, m in zip(splits[s], meta[s])]
report = diversity_report(records)
print(json.dumps({k: v for k, v in report.items() if k != "event_type_counts"}, indent=1))

STOP = set("the a an and or of to in on for is are was it this that with at by be as from has have not no its into than if".split())
words = collections.Counter(w for r in records for w in re.findall(r"[a-z]{4,}", r["user_text"].lower()) if w not in STOP)
causes = collections.Counter(w for s in splits for r in splits[s]
                             for w in re.findall(r"[a-z]{4,}", json.loads(r["messages"][2]["content"])["likely_cause"].lower()) if w not in STOP)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].hist([len(r["user_text"]) for r in records], bins=20); ax[0].set_title("Prompt length (chars)")
pd.Series(report["event_type_counts"]).plot.barh(ax=ax[1], title="Event type frequency")
pd.Series(dict(causes.most_common(15))).plot.barh(ax=ax[2], title="Top words in gold likely_cause")
plt.tight_layout(); plt.show()
print("Top prompt keywords:", words.most_common(25))

## 2B — QLoRA hyperparameters (every value justified)

| Parameter | Value | Reason |
| --- | --- | --- |
| Quantization | 4-bit NF4, double quant | NF4 is the QLoRA data type for normally distributed weights. Double quant saves about 0.4 bits per parameter, so the 3B model fits on a 15 GB T4 with room for activations. |
| Compute dtype | bf16 on L4/A100, fp16 on T4 | The T4 has no bf16 support. Detected at runtime. |
| LoRA rank `r` | 16 | The task is a fixed 6-key schema plus domain wording, so the adapter needs little capacity. With 120 training rows, r=32 adds parameters without adding signal and raises overfitting risk. |
| LoRA alpha | 32 | alpha/r = 2 scales the adapter update so it is noticeable against the frozen weights at lr 2e-4, the common QLoRA ratio. |
| LoRA dropout | 0.05 | Light regularisation for a small dataset; higher values slow convergence across only 3 epochs. |
| Target modules | q,k,v,o + gate,up,down | Attention changes what the model attends to in the record. MLP layers carry the domain vocabulary (quarantine, tender, ASN). Adapting both is the QLoRA-paper setting. |
| Learning rate | 2e-4 | The standard rate for LoRA adapters. Full fine-tune rates (about 2e-5) barely move a rank-16 adapter in 45 steps. |
| Scheduler | cosine, warmup ratio 0.03 | A short warmup avoids an early spike on a fresh adapter. Cosine decay lowers the LR in the last epoch, which helps validation loss settle rather than oscillate. |
| Epochs | 3 | 120 rows / effective batch 8 = 15 steps per epoch, 45 in total: enough to lock in the JSON format. More epochs on 120 rows risk memorising, and the val loss below checks this. |
| Batch size / grad accumulation | 1 / 8 (effective 8) | Batch 1 keeps peak memory low on a T4. Accumulating 8 gives a less noisy gradient. |
| Max sequence length | 768 tokens | Longest prompt is about 1,000 chars (roughly 250 tokens). Add the system prompt (~150) and the JSON answer (~200), and an example is around 600 tokens, so 768 avoids truncation. 1024+ only raises the OOM risk. The token-length cell below verifies this. |
| Optimiser | paged_adamw_8bit | Paged 8-bit states avoid optimiser-memory spikes (QLoRA recommendation). |
| Eval / save | every epoch, keep best by eval_loss | Gives the required per-epoch train/val loss and restores the best checkpoint if val loss rises. |
| Seed | 42 | Reproducibility. |

In [ ]:
import gc, torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), "| compute dtype:", DTYPE)
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

def train(model_id, max_len):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=BNB, device_map="auto", torch_dtype=DTYPE)
    model = get_peft_model(prepare_model_for_kbit_training(model), LORA)
    model.print_trainable_parameters()
    lengths = [len(tok.apply_chat_template(r["messages"], tokenize=True)) for r in splits["train"]]
    print(f"token lengths: max {max(lengths)}, mean {sum(lengths)/len(lengths):.0f} (max_length={max_len})")
    length_arg = "max_length" if "max_length" in SFTConfig.__dataclass_fields__ else "max_seq_length"
    args = SFTConfig(output_dir="outputs", num_train_epochs=3, per_device_train_batch_size=1, per_device_eval_batch_size=1,
                     gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
                     eval_strategy="epoch", logging_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                     load_best_model_at_end=True, metric_for_best_model="eval_loss", optim="paged_adamw_8bit",
                     bf16=BF16, fp16=not BF16, report_to="none", seed=42, **{length_arg: max_len})
    trainer = SFTTrainer(model=model, args=args, processing_class=tok,
                         train_dataset=Dataset.from_list(splits["train"]), eval_dataset=Dataset.from_list(splits["val"]))
    trainer.train()
    return tok, model, trainer

# OOM handling: shorter sequences first, then the 1.5B model. Every failure is logged.
ATTEMPTS = [("Qwen/Qwen2.5-3B-Instruct", 768), ("Qwen/Qwen2.5-3B-Instruct", 512), ("Qwen/Qwen2.5-1.5B-Instruct", 512)]
OOM_LOG = []
for MODEL_ID, MAX_LEN in ATTEMPTS:
    try:
        tok, model, trainer = train(MODEL_ID, MAX_LEN)
        break
    except torch.cuda.OutOfMemoryError as exc:
        OOM_LOG.append({"model": MODEL_ID, "max_length": MAX_LEN, "error": str(exc)[:200]})
        print("OOM ->", OOM_LOG[-1])
        gc.collect(); torch.cuda.empty_cache()
print("Trained:", MODEL_ID, "max_length", MAX_LEN, "| OOM log:", OOM_LOG or "no OOM encountered")

In [ ]:
hist = trainer.state.log_history
train_loss = [h["loss"] for h in hist if "loss" in h]
val_loss = [h["eval_loss"] for h in hist if "eval_loss" in h]
loss_df = pd.DataFrame({"epoch": range(1, len(val_loss) + 1), "train_loss": train_loss[: len(val_loss)], "val_loss": val_loss})
print(loss_df.to_string(index=False))
print("Validation loss decreases every epoch:", all(b < a for a, b in zip(val_loss, val_loss[1:])))
loss_df.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="Loss per epoch"); plt.show()

## 2C — Evaluation on the held-out test set
The base model is the same 4-bit Qwen with adapters **disabled** and the same system prompt. The fine-tuned model has adapters enabled. Both use greedy decoding on identical test rows.

In [ ]:
from task2_genai.src.schemas import AnomalyExplanation
from task2_genai.src.generate_dataset import extract_json
from task2_genai.src.evaluate import mean_rouge_l

def generate(messages, max_new_tokens=320):
    prompt = tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def valid(text):
    try: AnomalyExplanation.model_validate(extract_json(text)); return True
    except Exception: return False

model.eval(); model.config.use_cache = True
test = splits["test"]
gold = [r["messages"][2]["content"] for r in test]
with model.disable_adapter():
    base_preds = [generate(r["messages"]) for r in test]
tuned_preds = [generate(r["messages"]) for r in test]

results = pd.DataFrame({
    "model": ["Base Qwen + system prompt", "Fine-tuned (QLoRA)"],
    "ROUGE-L F1": [mean_rouge_l(base_preds, gold), mean_rouge_l(tuned_preds, gold)],
    "valid schema JSON": [sum(map(valid, base_preds)) / len(test), sum(map(valid, tuned_preds)) / len(test)],
    "test rows": [len(test)] * 2,
})
results

### Additional metric — LLM-as-judge (Groq `openai/gpt-oss-120b`, Pydantic-validated JSON)
The rubric is in `task2_genai/prompts/judge_system.txt`. It scores faithfulness (1–5), specificity (1–5), and a hallucinated flag. Invalid judge JSON is logged and retried once.

In [ ]:
import time
from groq import Groq
from task2_genai.src.evaluate import judge_one

client = Groq()
judged = {"base": [], "tuned": []}
for i, row in enumerate(test):
    for name, preds in (("base", base_preds), ("tuned", tuned_preds)):
        judged[name].append(judge_one(client, os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b"), row["messages"][1]["content"], gold[i], preds[i]))
        time.sleep(2)  # stay under the Groq free-tier rate limit

def summarise(scores):
    ok = [s for s in scores if s]
    return {"faithfulness": sum(s.faithfulness for s in ok) / len(ok), "specificity": sum(s.specificity for s in ok) / len(ok),
            "judge_hallucination_rate": sum(s.hallucinated for s in ok) / len(ok), "judged": f"{len(ok)}/{len(scores)}"}
pd.DataFrame({"Base": summarise(judged["base"]), "Fine-tuned": summarise(judged["tuned"])}).T

### Manual review — 10 fine-tuned responses
Read each response against its incident. Then set `REVIEW_LABELS` to `correct`, `partially_correct`, or `hallucinated`. The judge's flag is shown only as a hint.

In [ ]:
for i in range(10):
    print(f"===== #{i} =====\nINCIDENT:\n{test[i]['messages'][1]['content'][:700]}\n")
    print(f"FINE-TUNED:\n{tuned_preds[i]}\n\nGOLD:\n{gold[i]}")
    j = judged["tuned"][i]
    print(f"judge hint: hallucinated={j.hallucinated if j else 'n/a'} ({j.brief_reason if j else ''})\n")

In [ ]:
from task2_genai.src.evaluate import hallucination_rate
# Fill in after reading the ten responses above.
REVIEW_LABELS = ["", "", "", "", "", "", "", "", "", ""]
if all(REVIEW_LABELS):
    print(pd.Series(REVIEW_LABELS).value_counts().to_string())
    print(f"Hallucination rate: {hallucination_rate(REVIEW_LABELS):.0%} of {len(REVIEW_LABELS)} reviewed responses")
else:
    print("REVIEW_LABELS not filled in yet.")

### Qualitative analysis

**Where fine-tuning helped.** _Write after running. Cite specific test rows (#i) where the fine-tuned model produced valid JSON and a concrete, record-grounded action while the base model did not. Use the ROUGE-L, schema-validity, and judge numbers above._

**Remaining failure modes and next steps.** _Write after running. Name the failure pattern seen in the manual review (for example, generic actions on rare event types, or a wrong escalation flag). Name the data or training change that would address it (more examples of that event type, contrastive negatives, or loss on assistant tokens only)._

## Additional — RAG fallback when self-rated confidence is low
SOP snippets (`task2_genai/sop_corpus/sops.jsonl`) are embedded into an in-memory ChromaDB collection. If the model's `confidence` is below 0.6 (or missing), the top SOP is retrieved and the model is re-queried with it.

In [ ]:
import chromadb
from task2_genai.src.evaluate import parse_confidence

sops = [json.loads(line) for line in open("task2_genai/sop_corpus/sops.jsonl")]
collection = chromadb.Client().get_or_create_collection("sops")
collection.add(ids=[s["doc_id"] for s in sops], documents=[s["text"] for s in sops])
THRESHOLD = 0.6

confs = [parse_confidence(p) for p in tuned_preds]
i = min(range(len(test)), key=lambda k: -1 if confs[k] is None else confs[k])
row, before = test[i], tuned_preds[i]
print(f"Lowest-confidence test row #{i}: confidence={confs[i]} (threshold {THRESHOLD}) -> fallback {'TRIGGERED' if confs[i] is None or confs[i] < THRESHOLD else 'not needed; shown for illustration'}")
sop = collection.query(query_texts=[row["messages"][1]["content"]], n_results=1)["documents"][0][0]
after = generate([row["messages"][0], {"role": "user", "content": row["messages"][1]["content"] + "\n\nRelevant SOP:\n" + sop}])
print("\nRETRIEVED SOP:", sop, "\n\nBEFORE:", before, "\n\nAFTER:", after)
print(f"\nROUGE-L vs gold: before {mean_rouge_l([before], [gold[i]]):.3f} -> after {mean_rouge_l([after], [gold[i]]):.3f}; confidence after: {parse_confidence(after)}")

## Merge adapters and push the merged model to Hugging Face

In [ ]:
from huggingface_hub import whoami
from peft import PeftModel

trainer.save_model("adapters")  # best checkpoint (load_best_model_at_end)
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, "adapters").merge_and_unload()
merged.save_pretrained("merged_model"); tok.save_pretrained("merged_model")

HF_REPO_ID = f"{whoami(token=os.environ['HF_TOKEN'])['name']}/supply-chain-anomaly-{MODEL_ID.split('/')[-1].lower()}"
merged.push_to_hub(HF_REPO_ID, token=os.environ["HF_TOKEN"], private=False)
tok.push_to_hub(HF_REPO_ID, token=os.environ["HF_TOKEN"], private=False)
print("Merged model:", f"https://huggingface.co/{HF_REPO_ID}")

model = merged  # smoke test: the merged model answers one test incident
print(generate(test[0]["messages"]))